In [1]:

import sys
import subprocess
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):",
          round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

subprocess.run(["nvidia-smi"])


Python: 3.13.16
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
GPU memory (GB): 14.56


CompletedProcess(args=['nvidia-smi'], returncode=0)

## We'll simulate 20 compressor incidents, each containing temperature, vibration, and power readings.

In [2]:

import random
import json

random.seed(42)

incidents = []

for i in range(1, 21):
    incident = {
        "incident_id": f"INC-{i:03d}",
        "compressor_id": f"COMP-{i:02d}",
        "temperature_c": round(random.uniform(75, 105), 1),
        "vibration_mm_s": round(random.uniform(5, 14), 2),
        "power_kw": round(random.uniform(18, 30), 1),
        "status": "ANOMALY_DETECTED"
    }
    incidents.append(incident)

print("Total incidents:", len(incidents))
print(json.dumps(incidents[:3], indent=2))


Total incidents: 20
[
  {
    "incident_id": "INC-001",
    "compressor_id": "COMP-01",
    "temperature_c": 94.2,
    "vibration_mm_s": 5.23,
    "power_kw": 21.3,
    "status": "ANOMALY_DETECTED"
  },
  {
    "incident_id": "INC-002",
    "compressor_id": "COMP-02",
    "temperature_c": 81.7,
    "vibration_mm_s": 11.63,
    "power_kw": 26.1,
    "status": "ANOMALY_DETECTED"
  },
  {
    "incident_id": "INC-003",
    "compressor_id": "COMP-03",
    "temperature_c": 101.8,
    "vibration_mm_s": 5.78,
    "power_kw": 23.1,
    "status": "ANOMALY_DETECTED"
  }
]


## Load our LLM

In [3]:

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    device_map="cuda"
)

model.eval()

print("Model loaded successfully!")
print("Model:", model_id)
print("Device:", next(model.parameters()).device)


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded successfully!
Model: Qwen/Qwen2.5-0.5B-Instruct
Device: cuda:0


## Generate our first AI maintenance report

In [5]:

import json
import time

incident = incidents[0]


messages = [
    {
        "role": "system",
        "content": (
            "You are an industrial IoT maintenance reporting assistant. "
            "Your task is to generate concise, factual incident reports "
            "from simulated compressor sensor readings. "

            "Follow these strict rules: "
            "1. Report all sensor values exactly as provided. "
            "2. Do not classify readings as high, low, safe, or unsafe "
            "unless validated operating thresholds are supplied. "
            "3. Do not invent causes, diagnoses, or equipment faults. "
            "4. Never recommend shutdown, restart, or equipment control actions. "
            "5. An anomaly flag indicates an external detection system "
            "identified unusual behavior; it does not confirm a fault. "
            "6. Recommend reviewing sensor trends and consulting "
            "a qualified maintenance technician. "
            "7. Keep the report under 100 words."
        )
    },
    {
        "role": "user",
        "content": (
            "Generate a maintenance incident report using "
            "only the following sensor data:\n"
            + json.dumps(incident, indent=2)
            + "\n\nUse this structure:\n"
            "Incident ID:\n"
            "Compressor ID:\n"
            "Sensor Observations:\n"
            "Incident Status:\n"
            "Recommended Next Step:\n\n"
            "Do not include unsupported interpretations "
            "or operational control instructions."
        )
    }
]


prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to("cuda")

# Warm up the GPU before timing
with torch.inference_mode():
    model.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

torch.cuda.synchronize()
start = time.perf_counter()

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

torch.cuda.synchronize()
elapsed = time.perf_counter() - start

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]
report = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print("INCIDENT REPORT")
print("=" * 50)
print(report)

print("\nINFERENCE METRICS")
print("=" * 50)
print(f"Latency: {elapsed:.3f} seconds")
print(f"Generated tokens: {len(generated_tokens)}")
print(f"Output speed: {len(generated_tokens) / elapsed:.2f} tokens/sec")


INCIDENT REPORT
Incident ID: INC-001
Compressor ID: COMP-01
Sensor Observations:
Temperature: 94.2°C
Vibration: 5.23 mm/s
Power: 21.3 kW
Incident Status: ANOMALY_DETECTED

Recommend Reviewing Sensor Trends and Consulting Maintenance Technician.

INFERENCE METRICS
Latency: 2.452 seconds
Generated tokens: 74
Output speed: 30.18 tokens/sec


## Benchmark 20 incidents sequentially
### How long does our LLM take to generate 20 incident reports when processing them one at a time?

In [6]:

import time
import numpy as np

sequential_results = []

for incident in incidents:
    messages[1]["content"] = (
        "Generate a maintenance incident report using "
        "only the following sensor data:\n"
        + json.dumps(incident, indent=2)
        + "\n\nUse this structure:\n"
        "Incident ID:\n"
        "Compressor ID:\n"
        "Sensor Observations:\n"
        "Incident Status:\n"
        "Recommended Next Step:\n\n"
        "Do not include unsupported interpretations "
        "or operational control instructions."
    )

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to("cuda")

    torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    torch.cuda.synchronize()
    latency = time.perf_counter() - start

    generated_tokens = outputs[0][
        inputs["input_ids"].shape[1]:
    ]

    sequential_results.append({
        "incident_id": incident["incident_id"],
        "latency": latency,
        "tokens": len(generated_tokens)
    })

    print(
        f'{incident["incident_id"]}: '
        f'{latency:.2f}s | '
        f'{len(generated_tokens)} tokens'
    )

total_time = sum(r["latency"] for r in sequential_results)

print("\nSEQUENTIAL BENCHMARK RESULTS")
print("=" * 45)
print(f"Total time: {total_time:.2f} seconds")
print(f"Reports per minute: {len(incidents) / total_time * 60:.2f}")
print(f"Average latency: {np.mean([r['latency'] for r in sequential_results]):.2f}s")
print(f"P95 latency: {np.percentile([r['latency'] for r in sequential_results], 95):.2f}s")


INC-001: 2.37s | 74 tokens
INC-002: 2.58s | 75 tokens
INC-003: 3.11s | 75 tokens
INC-004: 2.37s | 74 tokens
INC-005: 2.38s | 74 tokens
INC-006: 2.33s | 74 tokens
INC-007: 2.50s | 74 tokens
INC-008: 3.02s | 74 tokens
INC-009: 2.42s | 75 tokens
INC-010: 2.44s | 75 tokens
INC-011: 2.36s | 75 tokens
INC-012: 2.47s | 75 tokens
INC-013: 3.01s | 75 tokens
INC-014: 2.38s | 75 tokens
INC-015: 2.35s | 71 tokens
INC-016: 2.34s | 74 tokens
INC-017: 2.37s | 74 tokens
INC-018: 3.07s | 73 tokens
INC-019: 2.40s | 75 tokens
INC-020: 2.36s | 74 tokens

SEQUENTIAL BENCHMARK RESULTS
Total time: 50.63 seconds
Reports per minute: 23.70
Average latency: 2.53s
P95 latency: 3.07s


## Prepare for continuous batching

## Test static batching
### We'll process all 20 incidents together

In [15]:

import time
import json
import torch

batch_prompts = []

for incident in incidents:
    batch_messages = [
        messages[0],
        {
            "role": "user",
            "content": (
                "Generate a maintenance incident report using "
                "only the following sensor data:\n"
                + json.dumps(incident, indent=2)
                + "\n\nUse this structure:\n"
                "Incident ID:\n"
                "Compressor ID:\n"
                "Sensor Observations:\n"
                "Incident Status:\n"
                "Recommended Next Step:\n\n"
                "Do not include unsupported interpretations "
                "or operational control instructions."
            )
        }
    ]

    batch_prompts.append(
        tokenizer.apply_chat_template(
            batch_messages,
            tokenize=False,
            add_generation_prompt=True
        )
    )

# Decoder-only models require left padding for batched generation.
tokenizer.padding_side = "left"

batch_inputs = tokenizer(
    batch_prompts,
    return_tensors="pt",
    padding=True
).to("cuda")

torch.cuda.synchronize()
start = time.perf_counter()

with torch.inference_mode():
    batch_outputs = model.generate(
        **batch_inputs,
        max_new_tokens=150,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

torch.cuda.synchronize()
batch_time = time.perf_counter() - start

input_length = batch_inputs["input_ids"].shape[1]

batch_reports = tokenizer.batch_decode(
    batch_outputs[:, input_length:],
    skip_special_tokens=True
)

print("STATIC BATCHING RESULTS")
print("=" * 45)
print(f"Total time: {batch_time:.2f} seconds")
print(f"Reports per minute: {20 / batch_time * 60:.2f}")
print(f"Speedup vs sequential: {50.63 / batch_time:.2f}x")
print(f"Generated reports: {len(batch_reports)}")

print("\nSAMPLE REPORT")
print("=" * 45)
print(batch_reports[0])


STATIC BATCHING RESULTS
Total time: 2.75 seconds
Reports per minute: 437.01
Speedup vs sequential: 18.44x
Generated reports: 20

SAMPLE REPORT
Incident ID: INC-001
Compressor ID: COMP-01
Sensor Observations:
Temperature: 94.2°C
Vibration: 5.23 mm/s
Power: 21.3 kW
Incident Status: ANOMALY_DETECTED

Recommend Reviewing Sensor Trends and Consulting Maintenance Technician.


## Validate all 20 generated reports

In [16]:

validation_results = []

for incident, report in zip(incidents, batch_reports):
    expected = [
        incident["incident_id"],
        incident["compressor_id"],
        str(incident["temperature_c"]),
        str(incident["vibration_mm_s"]),
        str(incident["power_kw"]),
        incident["status"]
    ]

    missing = [value for value in expected if value not in report]

    validation_results.append({
        "incident_id": incident["incident_id"],
        "passed": len(missing) == 0,
        "missing": missing
    })

passed = sum(r["passed"] for r in validation_results)

print("BATCH OUTPUT VALIDATION")
print("=" * 45)
print(f"Reports checked: {len(validation_results)}")
print(f"Reports passing field checks: {passed}/20")

for result in validation_results:
    if not result["passed"]:
        print(
            result["incident_id"],
            "Missing:",
            result["missing"]
        )


BATCH OUTPUT VALIDATION
Reports checked: 20
Reports passing field checks: 19/20
INC-015 Missing: ['19.0']


In [17]:
batch_reports[14]

'Incident ID: INC-015\nCompressor ID: COMP-15\nSensor Observations:\nTemperature: 81.8°C\nVibration: 7.6 mm/s\nPower: 19 kW\nIncident Status: ANOMALY_DETECTED\n\nRecommend Next Steps:\nReview sensor trends and consult a qualified maintenance technician for further analysis.'

## Quantization Technique
### Instead of storing numbers as high-precision 32-bit floats (FP32) or 16-bit floats (FP16), quantization maps them to lower-precision formats like 8-bit integers (INT8), 4-bit integers (INT4), or lower.
###  This process is similar to saving a high-resolution lossless image as a compressed JPEG

### Can we reduce the model's GPU memory consumption without significantly affecting inference performance?

In [19]:
#This installs the libraries needed to load the model with 8-bit weights
%pip install -q -U bitsandbytes accelerate


In [20]:
# Now we'll load a second version of Qwen2.5-0.5B-Instruct using 8-bit weight quantization.


import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Configure 8-bit weight quantization
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True
)

# Load the same model with INT8 weights
model_int8 = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct",
    quantization_config=quantization_config,
    device_map="cuda"
)

print("INT8 model loaded successfully!")
print("Device:", model_int8.device)
print("8-bit quantization:", model_int8.is_loaded_in_8bit)

# Check total GPU memory currently allocated
print(
    "Total allocated GPU memory:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

INT8 model loaded successfully!
Device: cuda:0
8-bit quantization: True
Total allocated GPU memory: 1.52 GB


## Measure FP16 vs INT8 model memory

In [21]:

def model_memory_mb(model):
    seen = set()
    total_bytes = 0

    for tensor in list(model.parameters()) + list(model.buffers()):
        storage = tensor.untyped_storage()
        key = (storage.device, storage.data_ptr())

        if key not in seen:
            seen.add(key)
            total_bytes += storage.nbytes()

    return total_bytes / (1024 ** 2)

fp16_memory = model_memory_mb(model)
int8_memory = model_memory_mb(model_int8)

print("MODEL MEMORY COMPARISON")
print("=" * 45)
print(f"FP16 model: {fp16_memory:.2f} MB")
print(f"INT8 model: {int8_memory:.2f} MB")
print(
    f"Memory reduction: "
    f"{(1 - int8_memory / fp16_memory) * 100:.2f}%"
)


MODEL MEMORY COMPARISON
FP16 model: 942.29 MB
INT8 model: 601.04 MB
Memory reduction: 36.21%


## We'll run the same 20 prompts as our earlier static-batching experiment.

In [22]:

import time
import torch

# Warm up the INT8 model
with torch.inference_mode():
    _ = model_int8.generate(
        input_ids=batch_inputs["input_ids"][:1],
        attention_mask=batch_inputs["attention_mask"][:1],
        max_new_tokens=10,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

torch.cuda.synchronize()
torch.cuda.reset_peak_memory_stats()

start = time.perf_counter()

with torch.inference_mode():
    int8_outputs = model_int8.generate(
        **batch_inputs,
        max_new_tokens=150,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

torch.cuda.synchronize()
int8_batch_time = time.perf_counter() - start

input_length = batch_inputs["input_ids"].shape[1]

int8_reports = tokenizer.batch_decode(
    int8_outputs[:, input_length:],
    skip_special_tokens=True
)

print("INT8 STATIC BATCHING RESULTS")
print("=" * 45)
print(f"Total time: {int8_batch_time:.2f} seconds")
print(f"Reports per minute: {20 / int8_batch_time * 60:.2f}")
print(f"FP16 batch time: {batch_time:.2f} seconds")
print(f"INT8 batch time: {int8_batch_time:.2f} seconds")
print(f"Speed ratio (FP16 / INT8): {batch_time / int8_batch_time:.2f}x")
print(f"Generated reports: {len(int8_reports)}")

print("\nSAMPLE INT8 REPORT")
print("=" * 45)
print(int8_reports[0])


Streaming output truncated to the last 5000 lines.


INT8 STATIC BATCHING RESULTS
Total time: 37.00 seconds
Reports per minute: 32.43
FP16 batch time: 2.75 seconds
INT8 batch time: 37.00 seconds
Speed ratio (FP16 / INT8): 0.07x
Generated reports: 20

SAMPLE INT8 REPORT
Incident ID: INC-001
Compressor ID: COMP-01
Sensor Observations:
Temperature: 94.2°C
Vibration: 5.23 mm/s
Power: 21.3 kW
Status: ANOMALY DETECTED

Recommended Next Step: Review sensor trends and consult a qualified maintenance technician.


## Verify the performance difference , let's repeat both models under identical conditions.

In [23]:

import time
import torch

def benchmark_model(model, inputs, runs=3):
    timings = []

    for i in range(runs):
        torch.cuda.synchronize()
        start = time.perf_counter()

        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=150,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        timings.append(elapsed)

        print(f"Run {i + 1}: {elapsed:.2f}s")

    return timings

print("FP16 BENCHMARK")
fp16_times = benchmark_model(model, batch_inputs)

print("\nINT8 BENCHMARK")
int8_times = benchmark_model(model_int8, batch_inputs)

print("\nCOMPARISON")
print("=" * 45)

fp16_median = sorted(fp16_times)[1]
int8_median = sorted(int8_times)[1]

print(f"FP16 median: {fp16_median:.2f}s")
print(f"INT8 median: {int8_median:.2f}s")
print(f"INT8 slowdown: {int8_median / fp16_median:.2f}x")


FP16 BENCHMARK
Run 1: 6.66s
Run 2: 2.70s


Run 3: 2.70s

INT8 BENCHMARK


Streaming output truncated to the last 5000 lines.


Run 1: 37.57s


Streaming output truncated to the last 5000 lines.


Run 2: 37.31s


Streaming output truncated to the last 5000 lines.


Run 3: 37.61s

COMPARISON
FP16 median: 2.70s
INT8 median: 37.57s
INT8 slowdown: 13.92x


## Check GPU utilization

In [24]:

import subprocess

result = subprocess.run(
    [
        "nvidia-smi",
        "--query-compute-apps=pid,process_name,used_gpu_memory",
        "--format=csv"
    ],
    capture_output=True,
    text=True
)

print("GPU PROCESSES")
print("=" * 45)
print(result.stdout)
print(result.stderr)


GPU PROCESSES
pid, process_name, used_gpu_memory [MiB]
1237, /usr/bin/python3, 2280 MiB




## Check individual benchmark timings

In [25]:

print("FP16 timings:", fp16_times)
print("INT8 timings:", int8_times)

print("\nGenerated token counts:")
print(
    "FP16:",
    [len(tokenizer.encode(r)) for r in batch_reports]
)
print(
    "INT8:",
    [len(tokenizer.encode(r)) for r in int8_reports]
)


FP16 timings: [6.65910052199979, 2.6971078229998966, 2.700124933999632]
INT8 timings: [37.57483367099985, 37.31350136099991, 37.60541730099976]

Generated token counts:
FP16: [73, 74, 74, 73, 73, 73, 73, 73, 74, 74, 74, 74, 74, 74, 77, 73, 73, 72, 74, 73]
INT8: [76, 74, 77, 79, 76, 72, 76, 79, 77, 77, 80, 77, 74, 76, 73, 76, 73, 75, 80, 72]
